In [ ]:
from flask import Flask, render_template, request
import pandas as pd
import numpy as np
import os
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from io import StringIO

app = Flask(__name__)
UPLOAD_FOLDER = "uploads"
PLOT_FOLDER = "static/plots"
os.makedirs(UPLOAD_FOLDER, exist_ok=True)
os.makedirs(PLOT_FOLDER, exist_ok=True)


@app.route("/")
def home():
    return render_template("home.html")


@app.route("/train")
def train():
    return render_template("train.html")
@app.route("/train_process", methods=["POST"])
def train_process():
    file = request.files["csv_file"]
    path = os.path.join(UPLOAD_FOLDER, file.filename)
    file.save(path)

    df = pd.read_csv(path)

    head = df.head().to_html(classes="table table-bordered")
    tail = df.tail().to_html(classes="table table-bordered")
    rows, cols = df.shape
    describe = df.describe().to_html(classes="table table-bordered")

    
    buffer = StringIO()
    df.info(buf=buffer)
    info = buffer.getvalue().replace("\n", "<br>")



    df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, format='mixed')
    df['Day'] = df['Date'].dt.day
    df['Month'] = df['Date'].dt.month
    df['Year'] = df['Date'].dt.year

    le = LabelEncoder()
    df['Company_ID'] = le.fit_transform(df['Company_ID'])
    df.drop(columns=['Date'], inplace=True)

    X = df.drop(columns=['Liquidity_Risk_Score'])
    y = df['Liquidity_Risk_Score']

    input_features = list(X.columns)
    output_label = "Liquidity_Risk_Score"

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42
    )

    split_info = {
        "X_train": X_train.shape,
        "X_test": X_test.shape,
        "y_train": y_train.shape,
        "y_test": y_test.shape
    }

   
    models = {
        "Linear Regression": LinearRegression(),
        "Random Forest": RandomForestRegressor(),
        "Gradient Boosting": GradientBoostingRegressor(),
        "SVR": SVR()
    }

    model_scores = {}

    for name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        model_scores[name] = round(r2_score(y_test, pred), 4)

        if name == "Random Forest":
            pickle.dump(model, open("model.pkl", "wb"))
            pickle.dump(scaler, open("scaler.pkl", "wb"))
            pickle.dump(le, open("encoder.pkl", "wb"))

   
    sns.histplot(y, kde=True)
    plt.savefig(f"{PLOT_FOLDER}/train_hist.png"); plt.close()

    sns.boxplot(y=y)
    plt.savefig(f"{PLOT_FOLDER}/train_box.png"); plt.close()

    sns.violinplot(y=y)
    plt.savefig(f"{PLOT_FOLDER}/train_violin.png"); plt.close()

    sns.heatmap(df.corr(), annot=True)
    plt.savefig(f"{PLOT_FOLDER}/train_corr.png"); plt.close()

    plt.scatter(df['Revenue_M$'], y)
    plt.savefig(f"{PLOT_FOLDER}/train_scatter.png"); plt.close()

    plt.bar(model_scores.keys(), model_scores.values())
    plt.savefig(f"{PLOT_FOLDER}/train_accuracy.png"); plt.close()

    return render_template(
        "train_result.html",
        head=head,
        tail=tail,
        rows=rows,
        cols=cols,
        describe=describe,
        info=info,
        input_features=input_features,
        output_label=output_label,
        split_info=split_info,
        model_scores=model_scores
    )
@app.route("/detect")
def detect():
    return render_template("detect.html")
@app.route("/detect_process", methods=["POST"])
def detect_process():
    model = pickle.load(open("model.pkl", "rb"))
    scaler = pickle.load(open("scaler.pkl", "rb"))
    encoder = pickle.load(open("encoder.pkl", "rb"))

    date = pd.to_datetime(request.form["date"], dayfirst=True, format='mixed')
    day, month, year = date.day, date.month, date.year

    revenue = float(request.form["revenue"])
    net_income = float(request.form["income"])
    eps = float(request.form["eps"])
    credit = float(request.form["credit"])
    operational = float(request.form["operational"])

    company = encoder.transform([request.form["company"]])[0]

    values = np.array([[company, revenue, net_income, eps,
                        credit, operational,
                        day, month, year]])

    scaled = scaler.transform(values)
    prediction = model.predict(scaled)[0]

    # ---- RISK LEVEL ----
    if prediction < 0.3:
        risk_level = "Low Liquidity Risk"
    elif prediction < 0.6:
        risk_level = "Medium Liquidity Risk"
    else:
        risk_level = "High Liquidity Risk"

  
    plt.bar(["Liquidity Risk"], [prediction]); plt.ylim(0,1)
    plt.savefig(f"{PLOT_FOLDER}/det_bar.png"); plt.close()

    plt.pie([prediction, 1-prediction], labels=["Risk","Safe"], autopct='%1.1f%%')
    plt.savefig(f"{PLOT_FOLDER}/det_pie.png"); plt.close()

    plt.bar(["Credit","Operational"], [credit, operational])
    plt.savefig(f"{PLOT_FOLDER}/det_risk.png"); plt.close()

    plt.violinplot(values.flatten())
    plt.savefig(f"{PLOT_FOLDER}/det_violin.png"); plt.close()

    plt.scatter([revenue], [prediction])
    plt.savefig(f"{PLOT_FOLDER}/det_scatter.png"); plt.close()

    plt.plot([0,1],[prediction,prediction])
    plt.savefig(f"{PLOT_FOLDER}/det_line.png"); plt.close()

    return render_template(
        "detect_result.html",
        prediction=round(prediction,3),
        risk_level=risk_level,
        revenue=revenue,
        net_income=net_income,
        eps=eps,
        credit=credit,
        operational=operational
    )


if __name__ == "__main__":
    app.run()


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [04/Feb/2026 16:00:06] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [04/Feb/2026 16:00:07] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [04/Feb/2026 16:00:08] "GET /static/home.jpg HTTP/1.1" 200 -
127.0.0.1 - - [04/Feb/2026 16:00:21] "GET /train HTTP/1.1" 200 -
127.0.0.1 - - [04/Feb/2026 16:00:21] "GET /static/train.jpg HTTP/1.1" 200 -
127.0.0.1 - - [04/Feb/2026 16:01:37] "POST /train_process HTTP/1.1" 200 -
127.0.0.1 - - [04/Feb/2026 16:01:37] "GET /static/plots/train_hist.png HTTP/1.1" 200 -
127.0.0.1 - - [04/Feb/2026 16:01:37] "GET /static/plots/train_violin.png HTTP/1.1" 200 -
127.0.0.1 - - [04/Feb/2026 16:01:37] "GET /static/plots/train_box.png HTTP/1.1" 200 -
127.0.0.1 - - [04/Feb/2026 16:01:37] "GET /static/plots/train_corr.png HTTP/1.1" 200 -
127.0.0.1 - - [04/Feb/2026 16:01:38] "GET /static/plots/train_accuracy.png HTTP/1.1" 200 -
127.0.0.1 - - [04/Feb/2026 16:01:38] "GET /static/plots/train_scatter.png HTTP/1.1